# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/octane21/flyrank-internship-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
import numpy as np
dataset = pd.read_csv('dataset.csv')


dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45019 entries, 0 to 45018
Data columns (total 77 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Unnamed: 0                       45019 non-null  int64  
 1   client_hash_id                   45019 non-null  object 
 2   content_hash_id                  45019 non-null  object 
 3   month                            45019 non-null  object 
 4   gsc_impressions_30d              45019 non-null  int64  
 5   gsc_clicks_30d                   45019 non-null  int64  
 6   gsc_sum_position_30d             45019 non-null  int64  
 7   ga4_pageviews_30d                45019 non-null  int64  
 8   ga4_sessions_30d                 45019 non-null  int64  
 9   ga4_users_30d                    45019 non-null  int64  
 10  ga4_engaged_sessions_30d         45019 non-null  int64  
 11  sessions_ai_30d                  45019 non-null  int64  
 12  sessions_organic_3

In [54]:
features = [
    'client_hash_id',
    'gsc_impressions_30d',
    'gsc_clicks_30d',
    'gsc_sum_position_30d',
    'ga4_pageviews_30d',
    'ga4_sessions_30d',
    'ga4_users_30d',
    'ga4_engaged_sessions_30d',
    'sessions_ai_30d',
    'sessions_organic_30d',
    'scroll_events_30d',
    'ga4_data_available',
    'gsc_data_available',
    'gsc_avg_position_30d',
    'gsc_impressions_prev_30d',
    'gsc_impressions_2m_ago_30d',
    'gsc_clicks_prev_30d',
    'gsc_clicks_2m_ago_30d',
    'gsc_sum_position_prev_30d',
    'gsc_sum_position_2m_ago_30d',
    'ga4_pageviews_prev_30d',
    'ga4_pageviews_2m_ago_30d',
    'ga4_sessions_prev_30d',
    'ga4_sessions_2m_ago_30d',
    'ga4_users_prev_30d',
    'ga4_users_2m_ago_30d',
    'ga4_engaged_sessions_prev_30d',
    'ga4_engaged_sessions_2m_ago_30d',
    'sessions_ai_prev_30d',
    'sessions_ai_2m_ago_30d',
    'sessions_organic_prev_30d',
    'sessions_organic_2m_ago_30d',
    'scroll_events_prev_30d',
    'scroll_events_2m_ago_30d',
    'gsc_avg_position_prev_30d',
    'gsc_avg_position_2m_ago_30d',
    'content_type',
    'search_volume',
    'competition',
    'competition_level',
    'cpc',
    'main_intent',
    'backlinks',
    'category_count',
    'word_count',
    'char_count',
    'content_age_days',
    'ctr_30d',
    'engagement_rate_30d',
    'scroll_rate_30d',
    'ai_traffic_30d',
    'decline_future_label'
]


dataset_90d = dataset[features].copy()

dataset_90d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45019 entries, 0 to 45018
Data columns (total 52 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   client_hash_id                   45019 non-null  object 
 1   gsc_impressions_30d              45019 non-null  int64  
 2   gsc_clicks_30d                   45019 non-null  int64  
 3   gsc_sum_position_30d             45019 non-null  int64  
 4   ga4_pageviews_30d                45019 non-null  int64  
 5   ga4_sessions_30d                 45019 non-null  int64  
 6   ga4_users_30d                    45019 non-null  int64  
 7   ga4_engaged_sessions_30d         45019 non-null  int64  
 8   sessions_ai_30d                  45019 non-null  int64  
 9   sessions_organic_30d             45019 non-null  int64  
 10  scroll_events_30d                45019 non-null  int64  
 11  ga4_data_available               45019 non-null  bool   
 12  gsc_data_available

In [78]:
from sklearn.model_selection import GroupKFold

X = dataset_90d.drop(
    columns=['decline_future_label']
)

y = dataset_90d['decline_future_label']
groups = dataset_90d['client_hash_id']

outer_gkf = GroupKFold(n_splits=5)

inner_gkf = GroupKFold(n_splits=4)

split_results = []

for outer_fold, (remaining_idx, test_idx) in enumerate(
    outer_gkf.split(X, y, groups=groups),
    start=1
):
    X_remaining = X.iloc[remaining_idx]
    y_remaining = y.iloc[remaining_idx]
    groups_remaining = groups.iloc[remaining_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    remaining_clients = groups_remaining.unique()

    inner_splits = list(
        inner_gkf.split(
            remaining_clients,
            groups=remaining_clients
        )
    )

    print(outer_fold)


X_test['client_hash_id'].unique()


1
2
3
4
5


array(['client_0e1acc6cd57b0eba', 'client_0fa64a184f18a4a0',
       'client_1a730cb2640a1abf', 'client_3197e6291363b4db',
       'client_3f0ce4d44fe94f3d', 'client_73cda7b4e4f265ea',
       'client_a2eeb8899886adde', 'client_a80fca3f171ed1de',
       'client_b10cb2997d0c7c86', 'client_c182d11e4862a37d',
       'client_f623b01661d4bfe4'], dtype=object)

In [72]:
len(dataset_90d['client_hash_id'].unique())

19

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.